# **UAP Trial lagi**

In [1]:
import pandas as pd
import pickle
import os
import random
import spacy
from string import punctuation

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.tag import pos_tag
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

### **Utility**

In [ ]:
stemmerx = SnowballStemmer('english')
lemmatizerx = WordNetLemmatizer()
stopx = stopwords.words('english')
data = pd.read_csv('./jobpostingdata.csv')
categories = {}

print(data.isnull().sum())
data = data.dropna()

X = data['text']
Y = data['fraudulent']

data.head()

title         0
fraudulent    0
text          0
dtype: int64


,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


### **Preprocessing**

In [23]:
def AlterTag (tag:str):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('R'):
        return 'r'
    return 'n'

def Preprocessing(docx: str):
    tokens = word_tokenize(docx.lower())
    tokens = [tok for tok in tokens if tok not in punctuation]
    tokens = [tok for tok in tokens if tok.isalpha()]
    tokens = [tok for tok in tokens if tok not in stopx]
    tokens = [tok for tok in tokens]

    tagged = pos_tag(tokens)
    tokens = [lemmatizerx.lemmatize(tok, AlterTag(tag)) for tok, tag in tagged]

    return tokens

### **Training**

In [4]:
def Training ():
    # Feats
    feats = []
    
    all_tokens = Preprocessing(' '.join(X))
    freqx = FreqDist(all_tokens)
    print('Distribution of Words:')
    print(freqx)

    for sent, label in zip(X,Y):
        clean = Preprocessing(sent)
        ft = {word: True for word in clean}
        feats.append([ft, label])

    random.shuffle(feats)

    # Split
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    # Train
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    # Info
    print('Model Trained')
    print(f'Accuracy: {(acc*100):.2f}%')
    print()

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    return model

def Load():
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        print('Model not Found!')
        print('Start Training...')
        model = Training()
        return model

In [ ]:
def Training2 ():
    # feats
    feats = []

    all_tokens = Preprocessing(' '.join(X))
    freqx = FreqDist(all_tokens)
    print('Most Common Data')
    print(freqx.most_common(10))

    for sent, label in zip(X, Y):
        clean = Preprocessing(sent)
        ft = {word: True for word in clean}
        feats.append([ft, label])

    # split
    split = int(len(feats) * 0.8)
    train_data = feats[:split]
    evals_data = feats[split:]

    # Train
    model = NaiveBayesClassifier.train(train_data)
    acc = accuracy(model, evals_data)

    # Info
    print('Model Trained')
    print(f'Accuracy: {acc*100}%')

    # Save
    with open('./model.pickle', 'wb') as file:
        pickle.dump(model, file)
    return model

def Load():
    if os.path.exists('./model.pickle'):
        with open('./model.pickle', 'rb') as file:
            model = pickle.load(file)
        return model
    else:
        model = Training2()
        return model

### **Embedding Language Model**

In [28]:
def TF_IDF (query: str):
    vectorizer = TfidfVectorizer(stop_words='english')
    mtx = vectorizer.fit_transform(X)
    qvec = vectorizer.transform([query])

    sim = cosine_similarity(mtx, qvec)
    data['Similarity'] = sim

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5 Job Recommendation for You')
    for i in range(5):
        print(f'{i+1}. {sorted_data.iloc[i,0]}')

def Ngrams (query: str):
    vectorizer = TfidfVectorizer(ngram_range=(1,3), stop_words='english')
    mtx = vectorizer.fit_transform(X)
    qvec = vectorizer.transform([query])

    sim = cosine_similarity(mtx, qvec)
    data['Similarity'] = sim

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5 Job Recommendation for You')
    for i in range(5):
        print(f'{i+1}. {sorted_data.iloc[i,0]}')

In [1]:
def Ngrams2 (query: str):
    vectorizer = TfidfVectorizer(ngram_range=(1,3), stop_words='english')
    mtx = vectorizer.fit_transform(X)
    qvec = vectorizer.transform([query])

    sim = cosine_similarity(mtx, qvec)
    data['Similarity'] = sim

    sorted_data = data.sort_values(by='Similarity', ascending=False)
    print('Top 5')
    for i in range(5):
        print(f'{1+i}. {sorted_data.iloc[i,0]}')


### **NER**

In [ ]:
def StartNER ():
    parg = ' '.join(X.head(100))
    ner = spacy.load('en_core_web_sm')
    parg = ner(parg)

    for ent in parg.ents:
        label = ent.label_
        if label not in categories:
            categories[label] = []
        categories[label].append(ent.text)


In [ ]:
def StartNER2 ():
    parg = ' '.join(X.head(100))
    ner = spacy.load('en_core_web_sm')
    parg = ner(parg)

    for ent in parg.ents:
        label = ent.label_
        if label not in categories:
            categories[label] = []
        categories[label].append(ent.text)

### **Support Function**

In [7]:
docxxx = None
categorex = None

In [18]:
def Enter():
    print('Press [Enter] to Continue...')
    input('')
    print('')
    print('')
    print('')
    print('')

def WriteText (model):
    global docxxx, categorex

    while True:
        print('Write Your Text: ')
        docxxx = input('')
        print('')
        
        if len(docxxx.split()) < 20:
            print('(!) Please Input at least 20 words')
            print('')
        else:
            print('Categorizing Text...')
            clean = Preprocessing(docxxx)
            feats = {word: True for word in clean}
            categorex = model.classify(feats)
            print('Categorizing Complete!\nText Saved!')
            break

def ViewRecomm ():
    if docxxx is None:
        print('Please Input your Text First')
        print('')
        return

    while True:
        print('Choose the Embedding Model:')
        print('1. TF-IDF')
        print('2. NGrams')
        cc = input('')

        if cc == '1':
            TF_IDF(docxxx)
            break
        elif cc == '2':
            Ngrams(docxxx)
            break
        else:
            print('Invalid Input')
    
def ViewNER ():
    if not categories:
        StartNER()
    
    for label, ent in categories.items():
        print(f'{label}: {", ".join(ent)}')


### **Main Menu**

In [30]:
def Menu():
    print('Loading Model...')
    model = Load()

    while True:
        print('')
        print('Ril or Fek')
        print('===')
        print(f'Your Text: {docxxx if docxxx != None else "Empty"}')
        print(f'Your Category: {("Fraud" if categorex == 1 else "Normal") if categorex != None else "Empty"}')
        print('1. Write Text')
        print('2. View Recommendation')
        print('3. View NER')
        print('4. Exit')
        print('>> ')
        cc = input('')
        print('')
        print('')

        if cc == '1':
            WriteText(model)
        elif cc == '2':
            ViewRecomm()
        elif cc == '3':
            ViewNER()
        elif cc == '4':
            break
        else:
            print('Invalid Input')

In [33]:
Menu()

Loading Model...

Ril or Fek
===
Your Text: 1 1 1 1 Software Engineer 1 1 1 1 1 1 1 1 1 Software Engineer 1 1 1 1 1 1 1 1 1 Software Engineer 1 1 1 1 1 1 1 1 1 Software Engineer 1 1 1 1 1 
Your Category: Normal
1. Write Text
2. View Recommendation
3. View NER
4. Exit
>> 


